In [ ]:
from collections import Counter
import json
import pandas as pd

from datetime import date
from sqlalchemy import ForeignKey, String, Date, Numeric, Integer, CheckConstraint
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, sessionmaker
from sqlalchemy import create_engine
import sqlite3

# Rellenamiento de la columna Group usando Inteligencia Artificial

## 1. Posibles listas de familias para la clasificación de los compuestos.

In [1]:
grupos_compuestos = {
    1: "Alcanos",
    2: "Alquenos y alquinos",
    3: "Ácidos carboxílicos",
    4: "Ésteres",
    5: "Alcoholes",
    6: "Fenoles",
    7: "Cetonas",
    8: "Aldehídos",
    9: "Éteres",
    10: "Aminas",
    11: "Amidas",
    12: "Nitrilos",
    13: "Anillos aromáticos",
    14: "Ésteres de ácido fosfórico",
    15: "Silanos / Siloxanos",
    16: "Esteroides / Terpenos",
    17: "Lactonas y lactamas",
    18: "Compuestos heterocíclicos",
    19: "Compuestos halogenados",
    20: "Otros",
    21: "Anilinas / Derivados de anilina",
    22: "Ftalatos",
    23: "Aldehídos aromáticos",
    24: "Cetonas aromáticas",
    25: "Amidas aromáticas",
    26: "Amidas alifáticas",
    27: "Alcoholes grasos / largos",
    28: "Ácidos grasos",
    29: "Compuestos organosulfurados",
    30: "Lactonas aromáticas",
    31: "Compuestos bencénicos sustituidos",
    32: "Compuestos nitrogenados heterocíclicos",
    33: "Compuestos con oxígeno cíclico",
    34: "Éteres de glicoles",
    35: "Compuestos con grupos TMS (trimetilsililo)",
    36: "Compuestos con anillos fundidos",
    37: "Compuestos policíclicos aromáticos (PAHs)",
    38: "Nitrilos aromáticos",
    39: "Otros"
}

In [2]:
grupos_amplios = {
    1: "Hidrocarburos (alcanos, alquenos, alquinos, PAHs)",
    2: "Ácidos carboxílicos y derivados (ésteres, anhídridos, sales)",
    3: "Alcoholes y fenoles (incluye alcoholes grasos)",
    4: "Cetonas y aldehídos (alifáticos y aromáticos)",
    5: "Éteres y compuestos oxigenados cíclicos (dioxanos, glicoles)",
    6: "Aminas y derivados nitrogenados (aminas, nitrilos, amidinas)",
    7: "Amidas y ureas (incluye lactamas y derivados aromáticos)",
    8: "Compuestos aromáticos (benceno, tolueno, sustituidos, anilinas)",
    9: "Compuestos heterocíclicos (piridinas, imidazoles, etc.)",
    10: "Organofosforados (fosfatos, fosfonatos)",
    11: "Organosulfurados (tioles, sulfonas, tiosulfonamidas)",
    12: "Compuestos halogenados (F, Cl, Br, I)",
    13: "Silanos y siloxanos (organosilicio)",
    14: "Esteroides, terpenos y lípidos complejos",
    15: "Otros (difíciles de clasificar o mixtos)"
}

## 2. Proceso de clasificación con diferentes Inteligencias Artificiales

Para obtener los archivos JSON con las clasificaciones realizadas por cada modelo, se siguió un proceso de consulta estructurado a dichos modelos.

En cada consulta, se proporcionaba al modelo la siguiente información:
* Un diccionario con los datos del compuesto (incluyendo la **fórmula**, el **nombre** y el **CAS**).
* Un diccionario con los **grupos de clasificación** disponibles.

Como resultado, el modelo devolvía un diccionario con el siguiente formato:  
`{CAS: número de grupo clasificado}`

Mediante este proceso, se generaron un total de **6 archivos JSON**, que contienen:
* 2 archivos por cada modelo, correspondientes a las clasificaciones en:
  - **Grupos amplios**  
  - **Grupos específicos**

## 3. Carga de las clasificaciones de IAs (ChatGPT, Gemini y Deepseek)

In [3]:
# CHATGPT
with open('../JSON/ChatGPT/compounds_clasificados_amplios_chatgpt.json', 'r', encoding='utf-8') as f:
    chatGPT_amplios_json = json.load(f)
with open('../JSON/ChatGPT/compounds_clasificados_especificos_chatgpt.json', 'r', encoding='utf-8') as f:
    chatGPT_especificos_json = json.load(f)

# GEMINI
with open('../JSON/Gemini/compounds_clasificados_amplios_gemini.json', 'r', encoding='utf-8') as f:
    gemini_amplios_json = json.load(f)  
with open('../JSON/Gemini/compounds_clasificados_especificos_gemini.json', 'r', encoding='utf-8') as f:
    gemini_especificos_json = json.load(f)

# DEEPSEEK
with open('../JSON/DeepSeek/compounds_clasificados_amplios_deepseek.json', 'r', encoding='utf-8') as f:
    deepseek_amplios_json = json.load(f)
with open('../JSON/DeepSeek/compounds_clasificados_especificos_deepseek.json', 'r', encoding='utf-8') as f:
    deepseek_especificos_json = json.load(f)

# ORIGINAL
with open('../JSON/Original/compounds.json', 'r', encoding='utf-8') as f:
    compuestos = json.load(f)

In [4]:
# PASAMOS LOS JSON A DICCIONARIOS QUE SOLO TIENEN 'CAS : NUMERO DE GRUPO' PARA HACER LA COMPARACIÓN MAS FACIL Y EFICIENTE
jsons = [
    chatGPT_amplios_json, 
    chatGPT_especificos_json, 
    gemini_amplios_json, 
    gemini_especificos_json, 
    deepseek_amplios_json, 
    deepseek_especificos_json]

chatGPT_amplios_clasificacion = {}
chatGPT_especificos_clasificacion = {}
gemini_amplios_clasificacion = {}
gemini_especificos_clasificacion = {}
deepseek_amplios_clasificacion = {}
deepseek_especificos_clasificacion = {}

diccionarios = [
    chatGPT_amplios_clasificacion, 
    chatGPT_especificos_clasificacion, 
    gemini_amplios_clasificacion, 
    gemini_especificos_clasificacion,
    deepseek_amplios_clasificacion,
    deepseek_especificos_clasificacion]

for file in jsons:
    diccionario = diccionarios.pop(0)
    for compuesto in file:
        diccionario[compuesto['cas']] = compuesto['group']


print(f"Tamaño de chatgpt_amplios = {len(chatGPT_amplios_clasificacion)}")
print(f"Tamaño de chatgpt_especificos = {len(chatGPT_especificos_clasificacion)}\n")

print(f"Tamaño de gemini_amplios = {len(gemini_amplios_clasificacion)}")
print(f"Tamaño de gemini_especificos = {len(gemini_especificos_clasificacion)}\n")

print(f"Tamaño de deepseek_amplios = {len(deepseek_amplios_clasificacion)}")
print(f"Tamaño de deepseek_especificos = {len(deepseek_especificos_clasificacion)}")

Tamaño de chatgpt_amplios = 9290
Tamaño de chatgpt_especificos = 9290

Tamaño de gemini_amplios = 9290
Tamaño de gemini_especificos = 9290

Tamaño de deepseek_amplios = 9290
Tamaño de deepseek_especificos = 9290


## 4. Comparación de las predicciones de los grupos amplios

In [5]:
iguales_amplios = 0
diferentes_amplios = 0

for compuesto in chatGPT_amplios_json:
    grupo_amplio_votado_chatgpt = chatGPT_amplios_clasificacion[compuesto['cas']]
    grupo_amplio_votado_gemini = gemini_amplios_clasificacion[compuesto['cas']]
    grupo_amplio_votado_deepseek = deepseek_amplios_clasificacion[compuesto['cas']]
    
    if grupo_amplio_votado_gemini == grupo_amplio_votado_chatgpt == grupo_amplio_votado_deepseek:
        iguales_amplios += 1
    else:
        diferentes_amplios += 1

print(f"{iguales_amplios} de {len(chatGPT_amplios_json)} han sido clasificados iguales")

2333 de 9290 han sido clasificados iguales


## 5. Comparación de las predicciones de los grupos especificos

In [6]:
iguales_especificos = 0
diferentes_especificos = 0

for compuesto in chatGPT_especificos_json:
    grupo_especifico_votado_chatgpt = chatGPT_especificos_clasificacion[compuesto['cas']]
    grupo_especifico_votado_gemini = gemini_especificos_clasificacion[compuesto['cas']]
    grupo_especifico_votado_deepseek = deepseek_especificos_clasificacion[compuesto['cas']]
    
    if grupo_especifico_votado_chatgpt == grupo_especifico_votado_gemini == grupo_especifico_votado_deepseek:
        iguales_especificos += 1
    else:
        diferentes_especificos += 1

print(f"{iguales_especificos} de {len(chatGPT_especificos_json)} han sido clasificados iguales")

1977 de 9290 han sido clasificados iguales


## 6. Creación de los grupos definitivos por votación (Ensemble de modelos)
### 6.1. Votacion de Grupos Amplios

In [7]:
grupos_amplios_ensemble = {}
acuerdos = 0

for compuesto in chatGPT_amplios_json:
    grupo_amplio_votado_chatgpt = chatGPT_amplios_clasificacion[compuesto['cas']]
    grupo_amplio_votado_gemini = gemini_amplios_clasificacion[compuesto['cas']]
    grupo_amplio_votado_deepseek = deepseek_amplios_clasificacion[compuesto['cas']]
    votos = [grupo_amplio_votado_chatgpt, grupo_amplio_votado_gemini, grupo_amplio_votado_deepseek]
    
    # Contar las apariciones de cada voto
    conteo_votos = Counter(votos)
    
    # Si cada uno a votado por un grupo diferente, se elige la opción votada por chatgpt
    if grupo_amplio_votado_chatgpt != grupo_amplio_votado_gemini and grupo_amplio_votado_chatgpt != grupo_amplio_votado_deepseek and grupo_amplio_votado_deepseek != grupo_amplio_votado_gemini:
        grupo_mayoritario = grupo_amplio_votado_chatgpt
    else:
        # Obtener el grupo con más votos (el de mayoría)
        grupo_mayoritario, _ = conteo_votos.most_common(1)[0]
        acuerdos += 1
    
    # Guardar la clasificación por mayoría
    grupos_amplios_ensemble[compuesto['cas']] = grupos_amplios[grupo_mayoritario]
    
print(f"Acuerdos totales: {acuerdos}")
grupos_amplios_ensemble

Acuerdos totales: 6963


{'4547-76-6': 'Ácidos carboxílicos y derivados (ésteres, anhídridos, sales)',
 '1000156-12-3': 'Otros (difíciles de clasificar o mixtos)',
 '1000132-73-7': 'Alcoholes y fenoles (incluye alcoholes grasos)',
 '64339-95-3': 'Ácidos carboxílicos y derivados (ésteres, anhídridos, sales)',
 '1000240-85-7': 'Aminas y derivados nitrogenados (aminas, nitrilos, amidinas)',
 '32634-68-7': 'Ácidos carboxílicos y derivados (ésteres, anhídridos, sales)',
 '86361-10-6': 'Alcoholes y fenoles (incluye alcoholes grasos)',
 '6831-16-9': 'Esteroides, terpenos y lípidos complejos',
 '1000375-69-4': 'Compuestos halogenados (F, Cl, Br, I)',
 '79815-20-6': 'Ácidos carboxílicos y derivados (ésteres, anhídridos, sales)',
 '1000156-14-3': 'Esteroides, terpenos y lípidos complejos',
 '1000352-28-0': 'Ácidos carboxílicos y derivados (ésteres, anhídridos, sales)',
 '34444-37-6': 'Otros (difíciles de clasificar o mixtos)',
 '1000156-11-8': 'Esteroides, terpenos y lípidos complejos',
 '7695-91-2': 'Ácidos carboxílico

### 6.2. Votacion de Grupos Especificos

In [8]:
grupos_especificos_ensemble = {}

acuerdos = 0

for compuesto in chatGPT_amplios_json:
    grupo_especifico_votado_chatgpt = chatGPT_especificos_clasificacion[compuesto['cas']]
    grupo_especifico_votado_gemini = gemini_especificos_clasificacion[compuesto['cas']]
    grupo_especifico_votado_deepseek = deepseek_especificos_clasificacion[compuesto['cas']]
    votos = [grupo_especifico_votado_chatgpt, grupo_especifico_votado_gemini, grupo_especifico_votado_deepseek]
    
    # Contar las apariciones de cada voto
    conteo_votos = Counter(votos)
    
    # Si cada uno a votado por un grupo diferente, se elige la opción votada por chatgpt
    if grupo_especifico_votado_chatgpt != grupo_especifico_votado_gemini and grupo_especifico_votado_chatgpt != grupo_especifico_votado_deepseek and grupo_especifico_votado_gemini != grupo_especifico_votado_deepseek:
        grupo_mayoritario = grupo_especifico_votado_chatgpt
    else:
        acuerdos += 1
        grupo_mayoritario, _ = conteo_votos.most_common(1)[0]
    
    # Guardar la clasificación por mayoría
    grupos_especificos_ensemble[compuesto['cas']] = grupos_compuestos[grupo_mayoritario]
print(f"Acuerdos totales: {acuerdos}")
grupos_especificos_ensemble

Acuerdos totales: 6765


{'4547-76-6': 'Ésteres',
 '1000156-12-3': 'Esteroides / Terpenos',
 '1000132-73-7': 'Amidas',
 '64339-95-3': 'Ácidos carboxílicos',
 '1000240-85-7': 'Aminas',
 '32634-68-7': 'Ácidos carboxílicos',
 '86361-10-6': 'Alcoholes',
 '6831-16-9': 'Esteroides / Terpenos',
 '1000375-69-4': 'Otros',
 '79815-20-6': 'Ácidos carboxílicos',
 '1000156-14-3': 'Esteroides / Terpenos',
 '1000352-28-0': 'Alcoholes grasos / largos',
 '34444-37-6': 'Otros',
 '1000156-11-8': 'Esteroides / Terpenos',
 '7695-91-2': 'Esteroides / Terpenos',
 '1000112-22-5': 'Otros',
 '1210-05-5': 'Aldehídos',
 '1000194-43-7': 'Aminas',
 '1000195-05-6': 'Aminas',
 '24008-11-5': 'Otros',
 '94883-96-2': 'Éteres',
 '94883-97-3': 'Éteres',
 '1000317-83-3': 'Aminas',
 '26187-27-9': 'Compuestos heterocíclicos',
 '7785-26-4': 'Otros',
 '131758-71-9': 'Compuestos organosulfurados',
 '13702-56-2': 'Alcoholes',
 '1000374-56-6': 'Éteres',
 '1000374-56-5': 'Éteres',
 '1000374-56-9': 'Éteres',
 '1000374-56-8': 'Éteres',
 '78349-00-5': 'Anill

## 7. Exportación de tablas a Excel

Para poder pasarle la clasificación a los responsables de la decisión, se exportaron a excel.

In [9]:
# Creamos el diccionario json sin los grupos (solo con name, cas y formula)
tabla_compuestos_amplios = compuestos.copy()

# Rellenamos el grupo de cada compuesto con los resultados de clasificación obtenidos
for compuesto in tabla_compuestos_amplios:
    compuesto['group'] = grupos_amplios_ensemble[compuesto['cas']]

# Guardamos la tabla completa en un archivo excel con CAS, Name, Formula y Group
compuestos_con_grupos_amplios = pd.DataFrame(tabla_compuestos_amplios)
compuestos_con_grupos_amplios.to_excel("../Excels Generados/Tabla_grupos_amplios_clasificados.xlsx", index=False)

In [11]:
# Lo mismo pero para los grupos especificos
tabla_compuestos_especificos = compuestos.copy()
for compuesto in tabla_compuestos_especificos:
    compuesto['group'] = grupos_especificos_ensemble[compuesto['cas']]
compuestos_con_grupos_especificos = pd.DataFrame(tabla_compuestos_especificos)
compuestos_con_grupos_especificos.to_excel("../Excels Generados/Tabla_grupos_especificos_clasificados.xlsx", index=False)

## 8. Insercción de los grupos en la base de datos

In [16]:
# Finalmente, se decidió por usar la clasificación de grupos especificos.

tabla_compuestos_especificos = pd.read_excel("../Excels Generados/Tabla_grupos_especificos_clasificados.xlsx")
tabla_compuestos_amplios = pd.read_excel("../Excels Generados/Tabla_grupos_amplios_clasificados.xlsx")

tabla_elegida = tabla_compuestos_especificos


engine = create_engine("sqlite:///../Database/Petrola.db")
Session = sessionmaker(bind=engine)
session = Session()

class Base(DeclarativeBase):
    pass

class Stations(Base):
    __tablename__ = "stations"
    
    station_id: Mapped[str] = mapped_column(String, primary_key=True)
    st_type: Mapped[str] = mapped_column(String(20), nullable=True)
    geology: Mapped[str] = mapped_column(String(20), nullable=True)
    x: Mapped[float] = mapped_column(Numeric(10,3), nullable=False)
    y: Mapped[float] = mapped_column(Numeric(10,3), nullable=False)
    
class Compounds(Base):
    __tablename__ = "compounds"
    
    cas: Mapped[str] = mapped_column(String(20), primary_key=True)
    name: Mapped[str] = mapped_column(String(80), nullable=False)
    formula: Mapped[str] = mapped_column(String(20), nullable=False)
    group: Mapped[str] = mapped_column(String(30), nullable=True)

class Samples(Base):
    __tablename__ = "samples"
    
    id: Mapped[int] = mapped_column(Integer, autoincrement=True, primary_key=True)
    station_id: Mapped[str] = mapped_column(ForeignKey("stations.station_id"), nullable=False)
    compound_cas: Mapped[str] = mapped_column(ForeignKey("compounds.cas"), nullable=False)
    component_rt: Mapped[float] = mapped_column(Numeric(17,13), nullable=False) #13 digitos en la parte decimal -> (0000.0000000000000 - 9999.9999999999999)
    library_rt: Mapped[float] = mapped_column(Numeric(17,13), nullable=True)
    match_factor: Mapped[float] = mapped_column(Numeric(16, 13), nullable=False) # como es un percentaje la parte entera admite solo 3 digitos y la decimal 13 digitos
    sample_date: Mapped[date] = mapped_column(Date, nullable=False)

    __table_args__ = (
        # No podemos insertar un porcentaje negativo o mayor a 100%
        CheckConstraint("match_factor >= 0 AND match_factor <= 100", name="check_match_factor_range"),
    )

Base.metadata.create_all(engine)


# Rellenamos el grupo
for _, compuesto in tabla_elegida.iterrows():
    compound = session.query(Compounds).filter(Compounds.cas == compuesto['cas']).first()
    if compound:
        compound.group = compuesto['group']
    else:
        print(f"No se encontró el compuesto con cas {compuesto['cas']}")

# Guardamos los cambios en el fichero petrola.db
session.commit()
session.close()
engine.dispose()

# Hacemos una consulta pequeña para ver si los grupos se han insertado correctamente
all_compounds = session.query(Compounds).all()
for cmp in all_compounds[:5]:
    print(f"{cmp.cas} - {cmp.name} - {cmp.formula} - {cmp.group}")

630-02-4 - Octacosane - C28H58 - Alcanos
3622-84-2 - Benzenesulfonamide, N-butyl- - C10H15NO2S - Compuestos organosulfurados
143-07-7 - Dodecanoic acid - C12H24O2 - Ácidos carboxílicos
84-66-2 - Diethyl phthalate - C12H14O4 - Ftalatos
57-11-4 - Octadecanoic acid - C18H36O2 - Ácidos carboxílicos


# 9. Creación Excel completo con grupos clasificados

Para cargar la base de datos completa con los grupos de forma fácil en cualquier momento, exportamos el dataframe a excel y lo guardamos

In [ ]:
tabla_compuestos_especificos = pd.read_excel("../Excels Generados/Tabla_grupos_especificos_clasificados.xlsx")
tabla_compuestos_amplios = pd.read_excel("../Excels Generados/Tabla_grupos_amplios_clasificados.xlsx")

# Comprobamos que la base de datos existe y realizamos la conexión
conn = sqlite3.connect("../Database/Petrola.db")

# Obtenemos los datos de las tres tablas (samples, stations y compounds) en formato dataframe
df_muestras = pd.read_sql_query("SELECT * FROM samples", conn)
df_stations = pd.read_sql_query("SELECT * FROM stations", conn)
df_compounds = pd.read_sql_query("SELECT * FROM compounds", conn)

# Cerramos la conexión con la base de datos
conn.close()

# Unimos los tres dataframes por sus foreign keys
df_muestras_stations = df_muestras.merge(df_stations, on='station_id', how='left')
df_petrola = df_muestras_stations.merge(df_compounds, left_on='compound_cas', right_on='cas', how='left')

# Añadimos la clasificación a la columna group de cada tupla
cas_to_group = tabla_compuestos_amplios.set_index('cas')['group']
df_petrola['group'] = df_petrola['compound_cas'].map(cas_to_group)

# Guardamos en excel
df_petrola.to_excel("../Excels Generados/df_petrola_con_grupos_amplios.xlsx", index=False)